# Chapter 09 — SQL Fundamentals and MySQL Connectivity

SQL is the language for relational data. Connect relational design and query logic to Python database drivers, transactions, parameterized SQL, and safe resource handling.

## Prerequisites
Basic tables and Python functions. MySQL server only needed for live-connection exercises; SQLite is used for local SQL drills.

## Learning goals
- Explain tables, rows, primary keys, and foreign keys.
- Write SELECT, WHERE, GROUP BY, HAVING, JOIN, and subqueries.
- Connect with mysql-connector-python or PyMySQL.
- Use parameterized statements, transactions, cursors, commits, and rollbacks.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. Relational model and keys
2. SELECT query anatomy
3. Joins and NULLs
4. Python MySQL drivers
5. Parameters, transactions, and security

### Relational model and keys
A table contains rows at a defined grain. A primary key uniquely identifies a row; a foreign key references a key in another table. A one-to-many relationship stores the parent key on the many-side. Normalization limits update anomalies. Define the grain and key before designing joins.

#### Demonstration 1 — Create local relational tables

Run the code and inspect both the output and the assumptions behind it.

In [2]:
import sqlite3
conn=sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys=ON")
conn.execute("CREATE TABLE customers(customer_id TEXT PRIMARY KEY, city TEXT NOT NULL)")
conn.execute("CREATE TABLE orders(order_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL REFERENCES customers(customer_id), revenue REAL NOT NULL CHECK(revenue>=0))")
conn.executemany("INSERT INTO customers VALUES (?,?)",[("C1","Kolkata"),("C2","Delhi")])
conn.executemany("INSERT INTO orders VALUES (?,?,?)",[("O1","C1",1200),("O2","C1",800),("O3","C2",1500)])
conn.commit(); print(conn.execute("SELECT * FROM orders").fetchall())

[('O1', 'C1', 1200.0), ('O2', 'C1', 800.0), ('O3', 'C2', 1500.0)]


### SELECT query anatomy
Typical SQL: `SELECT columns FROM table WHERE condition GROUP BY keys HAVING aggregate_condition ORDER BY expression LIMIT n`. Logical processing is conceptually FROM/JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT. `WHERE` filters rows; `HAVING` filters groups. `COUNT(*)` counts rows; `COUNT(column)` ignores NULL; `COUNT(DISTINCT column)` counts unique non-null values.

#### Demonstration 2 — Filter, join, group, and sort

Run the code and inspect both the output and the assumptions behind it.

In [3]:
sql="""SELECT c.city,COUNT(o.order_id) AS orders,SUM(o.revenue) AS revenue
FROM customers c JOIN orders o ON o.customer_id=c.customer_id
WHERE o.revenue>=? GROUP BY c.city HAVING COUNT(o.order_id)>=1 ORDER BY revenue DESC"""
print(conn.execute(sql,(500,)).fetchall())

[('Kolkata', 2, 2000.0), ('Delhi', 1, 1500.0)]


### Joins and NULLs
`INNER JOIN` keeps matching pairs; `LEFT JOIN` retains all left rows; `RIGHT JOIN` retains the right; `FULL OUTER JOIN` retains both where supported. Duplicate join keys can multiply rows. SQL uses three-valued logic: `column = NULL` is not a null test; use `IS NULL`/`IS NOT NULL`.

#### Demonstration 3 — Demonstrate safe parameterization

Run the code and inspect both the output and the assumptions behind it.

In [4]:
rows=conn.execute("SELECT order_id,revenue FROM orders WHERE customer_id=?",("C1",)).fetchall()
print(rows)
# Never interpolate user input into SQL with f-strings or string concatenation.

[('O1', 1200.0), ('O2', 800.0)]


### Python MySQL drivers
`mysql.connector.connect(host, port=3306, user, password, database, connection_timeout=...)` opens Connector/Python connections. `connection.cursor()` yields a cursor; `execute`, `fetchone`, `fetchmany(size)`, and `fetchall` are DB-API patterns. `cursor(dictionary=True)` returns mappings with Connector/Python. PyMySQL is another DB-API driver; its cursor classes include `DictCursor`.

#### Demonstration 4 — Demonstrate rollback

Run the code and inspect both the output and the assumptions behind it.

In [5]:
conn.execute("UPDATE orders SET revenue=? WHERE order_id=?",(9999,"O1")); conn.rollback()
print("O1 after rollback:",conn.execute("SELECT revenue FROM orders WHERE order_id=?",("O1",)).fetchone())
conn.close()

O1 after rollback: (1200.0,)


### Parameters, transactions, and security
Connector/Python and PyMySQL commonly use `%s` value placeholders; SQLite uses `?`. Pass values separately. Placeholders cannot safely stand for table or column identifiers; dynamic identifiers require allow-lists and proper quoting. `commit()` makes changes durable; `rollback()` cancels uncommitted work. Use try/except, close cursors/connections, environment variables, and least-privilege accounts.

#### Demonstration 5 — Show the MySQL driver pattern without connecting

Run the code and inspect both the output and the assumptions behind it.

In [6]:
import os
# import mysql.connector
# db=mysql.connector.connect(host=os.environ["MYSQL_HOST"],port=int(os.getenv("MYSQL_PORT","3306")),user=os.environ["MYSQL_USER"],password=os.environ["MYSQL_PASSWORD"],database=os.environ["MYSQL_DATABASE"],connection_timeout=10)
# try:
#     with db.cursor(dictionary=True) as cur:
#         cur.execute("SELECT order_id,revenue FROM orders WHERE region=%s",("North",))
#         records=cur.fetchall()
# finally: db.close()
print("Live MySQL is disabled until MYSQL_* variables and a database are configured.")

Live MySQL is disabled until MYSQL_* variables and a database are configured.


## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: Relational model and keys.
- Explain one important concept from: SELECT query anatomy.
- Explain one important concept from: Joins and NULLs.
- Explain one important concept from: Python MySQL drivers.
- Explain one important concept from: Parameters, transactions, and security.